# Encoding Categorical Data

**Notebook 4: Preparing Text Categories for Machine Learning**

## What you will learn

Many real datasets contain **categorical values**. These are values stored as text labels, such as:

- colour: `Red`, `Yellow`, `Blue`
- size: `Small`, `Medium`, `Large`
- fuel type: `gas`, `diesel`
- body style: `sedan`, `hatchback`, `wagon`

Many machine learning algorithms need numbers instead of text. Therefore, we often need to convert categorical values into numeric values before modelling. This process is called **encoding**.

By the end of this notebook, you should be able to:

- identify categorical columns in a DataFrame
- handle simple missing values in categorical data
- convert word-number categories using find-and-replace
- perform simple label encoding
- perform one-hot encoding using `pd.get_dummies()`
- explain when one-hot encoding is safer than label encoding

> **Important:** Encoding does not mean the category becomes mathematically bigger or smaller. It only changes the format so a computer can process it.

## 0. Quick idea: why encoding is needed

Imagine we have a column called `size` with these values:

| size |
|---|
| Small |
| Medium |
| Large |

A computer model may not understand these words directly. We can convert them into numbers, such as:

| size | encoded_size |
|---|---:|
| Small | 0 |
| Medium | 1 |
| Large | 2 |

This is easy to understand, but we must be careful. The model may think `Large = 2` is twice as much as `Medium = 1`. Sometimes that makes sense, but sometimes it does not.

That is why we will learn more than one encoding approach.

## 1. Dataset

In this lab, we will use an **Automobile Dataset** from the UCI Machine Learning Repository.

This dataset has no column headers in the original file, so we need to define the headers ourselves before loading it into Pandas.

Run the code below to load the dataset into a DataFrame called `df`.

In [ ]:
import pandas as pd
import numpy as np

# Define the headers since the data does not have any
headers = [
    "symboling", "normalized_losses", "make", "fuel_type", "aspiration",
    "num_doors", "body_style", "drive_wheels", "engine_location",
    "wheel_base", "length", "width", "height", "curb_weight",
    "engine_type", "num_cylinders", "engine_size", "fuel_system",
    "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm",
    "city_mpg", "highway_mpg", "price"
]

# Read in the file and convert "?" to NaN
df = pd.read_csv("./data/imports-85.data", header=None, names=headers, na_values="?")

# Show the first 5 rows
df.head()

## 1.1 Check the data types

Before encoding categorical columns, we need to know which columns contain text values.

Use the `.dtypes` attribute to view the data type of each column.

> **Expected output:** Some columns should show as `object` or string (`str`) columns. These are likely categorical columns.

In [ ]:
# Show the data type of each column
df.______

## 1.2 Select only categorical columns

Since this notebook focuses on encoding categorical variables, we will create a new DataFrame called `obj_df` that contains only the text/categorical columns.

Use `select_dtypes()` to select columns with `object` data type.

> **Hint:** In many Pandas versions, text columns are selected using `include=["object", "str"]`.

In [ ]:
# Select only categorical/text columns
obj_df = df.select_dtypes(include=["______"]).copy()

# Show the first 5 rows
obj_df.head()

## 1.3 Check for missing categorical values

Before encoding, we should check whether any categorical columns have missing values.

The code below shows rows where at least one categorical value is missing.

In [ ]:
obj_df[obj_df.isnull().any(axis=1)]

You should notice that the `num_doors` column has missing values.

Before filling the missing values, let us find the most common value in the `num_doors` column.

In [ ]:
# Count how many times each value appears in num_doors
obj_df["num_doors"].________()

Since `four` is the most common value, we will fill the missing values in `num_doors` with `four`.

> **Checkpoint:** After filling, the check should show no rows with missing categorical values.

In [ ]:
# Fill missing values in num_doors with the most common value
obj_df["num_doors"] = obj_df["num_doors"].fillna("____")

# Check again to ensure no more null values in categorical columns
obj_df[obj_df.isnull().any(axis=1)]

## 2. Approach #1 – Find and Replace

The first approach is simple replacement. We replace text values with their numeric equivalents.

This works well when the text values already represent numbers.

In this dataset, two columns are good examples:

- `num_doors`: `two`, `four`
- `num_cylinders`: `two`, `three`, `four`, `five`, `six`, `eight`, `twelve`

First, use `value_counts()` to explore the values in `num_cylinders`.

In [ ]:
# Count values in num_cylinders
obj_df["________"].value_counts()

The `replace()` method can use a Python dictionary to map old values to new values.

The example below converts `num_doors` from words into numbers.

In [ ]:
cleanup_num_doors = {"num_doors": {"four": 4, "two": 2}}

obj_df.replace(cleanup_num_doors, inplace=True)
obj_df["num_doors"] = obj_df["num_doors"].astype("int64")
obj_df.head()

Now it is your turn. Convert the string values in `num_cylinders` into their numeric values.

> **Hint:** Use a dictionary similar to the previous example.

In [ ]:
# Convert num_cylinders from words into numbers
cleanup_num_cylinders = {
    "num_cylinders": {
        "two": ____,
        "three": ____,
        "four": ____,
        "five": ____,
        "six": ____,
        "eight": ____,
        "twelve": ____
    }
}

obj_df.replace(________, inplace=True)
obj_df[_______] = obj_df["num_cylinders"].____("___")
obj_df.head()

Check the data types again. Pandas should now recognise `num_doors` and `num_cylinders` as numeric columns instead of text columns.

In [ ]:
obj_df.dtypes

### When is find-and-replace useful?

This approach is useful when the category has an obvious numeric meaning.

Examples:

- `two` → `2`
- `four` → `4`
- `low`, `medium`, `high` → maybe `1`, `2`, `3` if the order is meaningful

However, it is not suitable for every category. For example, converting `sedan`, `hatchback`, and `wagon` into `0`, `1`, and `2` may accidentally create a fake ranking.

## 3. Approach #2 – Label Encoding

Label encoding converts each category into a number.

For example, the `body_style` column contains values such as:

- `convertible`
- `hardtop`
- `hatchback`
- `sedan`
- `wagon`

We could encode them as numbers like this:

| body_style | encoded value |
|---|---:|
| convertible | 0 |
| hardtop | 1 |
| hatchback | 2 |
| sedan | 3 |
| wagon | 4 |

One simple Pandas method is to convert a column to the `category` data type, then use `.cat.codes`.

First, convert the `body_style` column into a category using `.astype("category")`.

In [ ]:
# Convert body_style to category data type
obj_df["body_style"] = obj_df["body_style"].astype("________")

# Check the data types
obj_df.dtypes

Now assign the encoded values to a new column called `body_style_cat` using `.cat.codes`.

> **Expected output:** You should see a new numeric column named `body_style_cat`.

In [ ]:
# Create a new encoded column
obj_df["body_style_cat"] = obj_df["body_style"].cat.________

obj_df.head()

### Important limitation of label encoding

Label encoding is straightforward, but it can mislead some machine learning algorithms.

For example, if:

- `convertible = 0`
- `sedan = 3`
- `wagon = 4`

the algorithm may think `wagon` is greater than `sedan`, or four times `convertible`. For categories with no natural order, that meaning may be wrong.

That is why we often use one-hot encoding instead.

## 4. Approach #3 – One-Hot Encoding

One-hot encoding converts each category into a new column. Each new column stores `1` or `0` to show whether the row belongs to that category.

For example, the `drive_wheels` column has values such as:

- `4wd`
- `fwd`
- `rwd`

One-hot encoding creates columns such as:

- `drive_4wd`
- `drive_fwd`
- `drive_rwd`

This avoids creating a fake ranking between categories.

Pandas provides `pd.get_dummies()` to perform one-hot encoding.

Run the example below. It one-hot encodes the `drive_wheels` column.

In [ ]:
pd.get_dummies(obj_df, columns=["drive_wheels"], prefix=["drive"]).head()

The new dataset contains new columns such as:

- `drive_4wd`
- `drive_fwd`
- `drive_rwd`

Each row will have a `True` in the matching category column and `False` in the others.

Now it is your turn. Perform one-hot encoding on both `body_style` and `drive_wheels`.

Requirements:

- encode columns `body_style` and `drive_wheels`
- use prefixes `body` and `drive`
- assign the new DataFrame to `obj_df2`
- show the first 5 rows

> **Hint:** Use `pd.get_dummies(dataframe, columns=[...], prefix=[...])`.

In [ ]:
obj_df2 = pd.get_dummies(
    obj_df,
    columns=["________", "________"],
    prefix=["________", "________"]
)

obj_df2.head()

## 5. Comparing the three approaches

| Approach | Good for | Be careful when |
|---|---|---|
| Find and replace | text values that clearly mean numbers | categories do not have numeric meaning |
| Label encoding | quick conversion into numbers | categories have no natural order |
| One-hot encoding | categories with no ranking | there are many unique categories, causing many new columns |

One-hot encoding is very useful, but it can greatly increase the number of columns if a category has many unique values.

## 6. Reflection questions

Answer these questions in your own words.

In [ ]:
# Write your answers as Python comments below.
# 1. Why do we need to encode categorical data before machine learning?
# 2. When is find-and-replace a suitable method?
# 3. What is one possible problem with label encoding?
# 4. Why is one-hot encoding often safer for categories such as body_style?
# 5. What is one disadvantage of one-hot encoding?


## Recap

In this notebook, you practised how to:

- identify categorical columns
- fill missing categorical values
- use `value_counts()` to inspect categories
- replace word-number categories with numeric values
- use `.astype("category")` and `.cat.codes` for label encoding
- use `pd.get_dummies()` for one-hot encoding

You have now completed the main data preparation steps in Chapter 1:

```text
Get data → Understand data → Visualise data → Clean data → Encode data
```

These steps prepare a dataset so that it can be used for machine learning later.